# Cross-validation

Differential expression and gene selection are fitted separately on each training fold only. The resulting genes are then used to subset that fold's training and validation data, avoiding feature-selection leakage into validation.

## 1. Train & test definition

We set a 20% test 80% train

We use random seed for reproducible results

In [8]:
import sys
import pandas as pd

sys.path.append('..')  # repository root when running from notebooks/

from src.dea import run_differential_expression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

from src.dea import (
    run_differential_expression
)

In [2]:
data_dir = '../data/processed'

#import datasets
counts_filtered = pd.read_csv(
    f'{data_dir}/raw_merged_filtered.csv',
    index_col=0,
)
metadata = pd.read_csv(
    f'{data_dir}/metadata_merged.csv',
    index_col=0,
)

#Find common ids (samples) betwwen genes and metadata
common_ids = counts_filtered.index.intersection(metadata.index)

#Define features (X) and target variable (y)
X = counts_filtered.loc[common_ids]
y = metadata.loc[common_ids, 'Condition']

#Check alignment
assert X.index.equals(y.index)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y 
)

In [4]:
print("Training:", X_train.shape)
print("Test:", X_test.shape)

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Training: (185, 27249)
Test: (47, 27249)

Training distribution:
Condition
Cancer     95
Healthy    90
Name: count, dtype: int64

Test distribution:
Condition
Cancer     24
Healthy    23
Name: count, dtype: int64


In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [11]:
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train)
):

    # Split the samples
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]

    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    _, selected_genes = run_differential_expression(
        X_fold_train,
        y_fold_train,
    )

    X_fold_train_selected = X_fold_train.loc[:, selected_genes]
    X_fold_val_selected = X_fold_val.loc[:, selected_genes]

    print("selected_genes:", len(selected_genes))
    print("Training shape:", X_fold_train_selected.shape)
    print("Validation shape:", X_fold_val_selected.shape)

    print(f"\nFold {fold + 1}:")

    print("Train:", X_fold_train.shape)
    print("Validation:", X_fold_val.shape)

    print(y_fold_train.value_counts())
    print(y_fold_val.value_counts())

Fitting size factors...
... done in 0.10 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 4.74 seconds.

Fitting dispersion trend curve...
... done in 0.33 seconds.

Fitting MAP dispersions...
... done in 5.23 seconds.

Fitting LFCs...
... done in 2.74 seconds.

Calculating cook's distance...
... done in 0.19 seconds.

Replacing 2744 outlier genes.

Fitting dispersions...
... done in 0.49 seconds.

Fitting MAP dispersions...
... done in 0.50 seconds.

Fitting LFCs...
... done in 0.35 seconds.

Running Wald tests...
... done in 3.45 seconds.

Fitting MAP LFCs...


Log2 fold change & Wald test p-value: Condition Cancer vs Healthy
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG         15.182657        0.472264  0.137768  0.000000  1.000000  1.000000
A1BG-AS1     93.604683        0.542958  0.111900  0.000000  1.000000  1.000000
A1CF          5.439061        0.319166  0.323470  0.000000  1.000000  1.000000
A2M       74998.189750       -0.908396  0.122940  0.000000  1.000000  1.000000
A2M-AS1     161.788336       -1.224865  0.119437 -1.882710  0.059740  0.393105
...                ...             ...       ...       ...       ...       ...
ZYG11A      313.139702        1.872992  0.198313  4.402085  0.000011  0.000148
ZYG11B     4030.691762       -0.245620  0.091772  0.000000  1.000000  1.000000
ZYX       11259.043056       -0.367700  0.104027  0.000000  1.000000  1.000000
ZZEF1      5502.510645       -0.259002  0.066835  0.000000  1.000000  1.000000
ZZZ3       4180.231835       -0.007582  0.073467  0.000000  1.000

... done in 1.75 seconds.



Shrunk log2 fold change & Wald test p-value: Condition[T.Cancer]
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG         15.182657        0.458932  0.136832  0.000000  1.000000  1.000000
A1BG-AS1     93.604683        0.533394  0.111549  0.000000  1.000000  1.000000
A1CF          5.439061       -0.429013  0.309179  0.000000  1.000000  1.000000
A2M       74998.189750       -0.880770  0.122633  0.000000  1.000000  1.000000
A2M-AS1     161.788336       -1.211255  0.119503 -1.882710  0.059740  0.393105
...                ...             ...       ...       ...       ...       ...
ZYG11A      313.139702        1.841086  0.198735  4.402085  0.000011  0.000148
ZYG11B     4030.691762       -0.472868  0.091578  0.000000  1.000000  1.000000
ZYX       11259.043056       -0.356883  0.103596  0.000000  1.000000  1.000000
ZZEF1      5502.510645       -0.257338  0.066719  0.000000  1.000000  1.000000
ZZZ3       4180.231835       -0.008725  0.073288  0.000000  1.0000

... (19 outputs left out)

Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 4.77 seconds.

Fitting dispersion trend curve...
... done in 0.32 seconds.

Fitting MAP dispersions...
... done in 5.40 seconds.

Fitting LFCs...
... done in 2.67 seconds.

Calculating cook's distance...
... done in 0.19 seconds.

Replacing 2676 outlier genes.

Fitting dispersions...
... done in 0.47 seconds.

Fitting MAP dispersions...
... done in 0.51 seconds.

Fitting LFCs...
... done in 0.35 seconds.

Running Wald tests...
... done in 3.30 seconds.

Fitting MAP LFCs...


Log2 fold change & Wald test p-value: Condition Cancer vs Healthy
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG         15.612086        0.443920  0.133813  0.000000  1.000000  1.000000
A1BG-AS1     92.871334        0.516874  0.108747  0.000000  1.000000  1.000000
A1CF          5.638791       -0.065836  0.299954  0.000000  1.000000  1.000000
A2M       73603.315011       -0.987956  0.108486  0.000000  1.000000  1.000000
A2M-AS1     158.981409       -1.238898  0.118732 -2.012071  0.044212  0.300960
...                ...             ...       ...       ...       ...       ...
ZYG11A      278.674979        1.635921  0.196291  3.239692  0.001197  0.011595
ZYG11B     3970.534296       -0.252798  0.091505  0.000000  1.000000  1.000000
ZYX       11248.755977       -0.430496  0.101644  0.000000  1.000000  1.000000
ZZEF1      5360.153765       -0.276937  0.064971  0.000000  1.000000  1.000000
ZZZ3       4112.138674       -0.036928  0.070676  0.000000  1.000

... done in 1.89 seconds.

